# Static augmentation

## Overview

This workflow uses a **"static" data augmentation workflow** for vertebrallandmark localization in sagittal spine MRI.  
**All augmentations are applied at dataset level** and remain **fixed across training**, ensuring stable optimization and reproducibility.

The model is trained to predict **heatmaps** for predefined anatomical levels (e.g. L1/L2 – L5/S1), from which landmark coordinates are extracted via argmax.

## Augmentation Strategy

Augmentations are applied **only to the training dataset**.  
Validation and test datasets remain **unaltered**.

### 1. Paired Geometric Augmentations (Image + Heatmaps)

These transformations are applied **identically** to the image and its corresponding heatmaps to preserve spatial alignment:

- Random rotation (extended range, e.g. ±15°)
- Random translation (percentage of image size)
- Random scaling
- Random shear
- Horizontal flip
- Vertical flip

This ensures that landmark supervision remains geometrically correct after transformation.

---

### 2. Image-only Intensity Augmentations

These transformations affect **only the input image**, not the heatmaps:

- Brightness adjustment
- Contrast adjustment
- Gamma correction
- Gaussian blur
- Optional noise (if enabled)

Intensity augmentations simulate acquisition variability across scanners and protocols.

---

## Training Workflow

1. Dataset is constructed with static augmentation enabled for training.
2. Model is trained using:
   - Heatmap regression loss (e.g. weighted MSE)
   - Automatic Mixed Precision (AMP) if GPU is available
3. Early stopping is applied based on validation loss.
4. Best-performing model weights are restored in memory.
5. Training and validation losses are visualized after training.

Optional schedules (sigma reduction or resolution increase) can be enabled, but the **augmentation itself remains static**.

---


## Data preprocessing

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
import cv2
import re
from PIL import Image
import torch
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from tqdm import tqdm
from torch.utils.data import random_split, DataLoader
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
%run ./Pretraining-datasets.ipynb

In [ ]:
%run ./Pretraining-models.ipynb

In [ ]:
%run ./Pretraining-fce.ipynb

In [ ]:
BASE_DIR = Path.cwd()
print(BASE_DIR)

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"       # quieter logs (optional)
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"  # enables growth mode
torch.cuda.is_available(), torch.cuda.get_device_name(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df_coords = pd.read_csv("df_coords.csv")
print("Loaded df_coords:")
display(df_coords)

In [ ]:
path = './data/processed_osf_jpgs/ID39.jpg'
img = Image.open(path)
print(img.size) 

In [ ]:
# Normalize for training
IMG_SIZE = 256
mean = [0.2127]
std = [0.2673]

img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

ds = SpineLandmarksDataset(
    df_coords,
    img_size=256,
    sigma=5,
    transform=img_transform,
)

print("Number of images:", len(ds))
print("Levels / channels:", ds.levels)

img, target = ds[0]
print("img.shape    =", img.shape)
print("target.shape =", target.shape)

## Dataset preparation

In [ ]:
# ---- Split (no leakage) ----
df_train, df_val, df_test = split_by_scan(df_coords, val_frac=0.2, seed=42)


In [ ]:
train_ds = SpineLandmarksDatasetStaticAug(
    df_train,
    img_size=IMG_SIZE,
    sigma=5.0,
    transform=img_transform,
    augment=True,
    rot_deg=15.0
)

val_ds = SpineLandmarksDatasetStaticAug(
    df_val,
    img_size=IMG_SIZE,
    sigma=5.0,
    transform=img_transform,
    augment=False
)

test_ds = SpineLandmarksDatasetStaticAug(
    df_test,
    img_size=IMG_SIZE,
    sigma=5.0,
    transform=img_transform,
    augment=False
)

print(f"Train samples: {len(train_ds)}, Val samples: {len(val_ds)}, Teat samples: {len(test_ds)}")

In [ ]:
# DataLoaders
BATCH_SIZE = 8
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=False
)

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))
print("Levels:", train_ds.levels)

In [ ]:
visualize_training_batch(train_loader, mean=mean, std=std, level_idx=2)

## Models training

In [ ]:
# CONFIG: 

class WeightedMSELoss(nn.Module):
    def __init__(self, alpha=10.0):
        super().__init__()
        self.alpha = float(alpha)

    def forward(self, pred, target):
        w = 1.0 + self.alpha * target
        return (w * (pred - target) ** 2).mean()

num_epochs = 30
patience   = 10
criterion = WeightedMSELoss(alpha=10.0)

# --- optional schedules ---
dynamic_sigma_reduction = True
resolution_increasing   = False

# sigma schedule (only used if dynamic_sigma_reduction=True)
sigma_start = 10.0
sigma_end   = 3.0

# resolution schedule (only used if resolution_increasing=True) -> NOT APPLIED 
res_start = 128
res_end   = 256

# AMP
use_amp = torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_val_loss = math.inf
best_state = None
epochs_without_improve = 0

# convenience: dataset refs (must have set_sigma/set_img_size)
train_ds_ref = train_loader.dataset
val_ds_ref   = val_loader.dataset


### U-Net

In [ ]:
model1 = UNet(
    in_channels=1,   # grayscale
    n_classes=5,
).to(device)

optimizer1 = torch.optim.Adam(model1.parameters(), lr=1e-3, weight_decay=1e-4)

print("Model ready on", device)

total_params = sum(p.numel() for p in model1.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
model1, history1 = training_static_aug(
    model=model1,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer1,
    device=device,
    criterion=criterion,
    num_epochs=30,
    patience=10,
    dynamic_sigma_reduction=False,
    resolution_increasing=False,
)

### Res-U-Net

In [ ]:
model2 = ResUNet(
    in_channels=1,   # grayscale
    n_classes=5,
    base_c=64        # matches the diagram; you can use 32 if you want smaller model
).to(device)

optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3, weight_decay=1e-4)

print("Model ready on", device)

total_params = sum(p.numel() for p in model2.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
model2, history2 = training_static_aug(
    model=model2,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer2,
    device=device,
    criterion=criterion,
    num_epochs=30,
    patience=10,
    dynamic_sigma_reduction=False,
    resolution_increasing=False,
)


### Attention Res U-Net

In [ ]:
model3 = AttentionResUNet(
    in_channels=1,   # grayscale
    n_classes=5,
    base_c=64  
).to(device)

optimizer3 = torch.optim.Adam(model3.parameters(), lr=1e-3, weight_decay=1e-4)

print("Model ready on", device)

total_params = sum(p.numel() for p in model3.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
model3, history3 = training_static_aug(
    model=model3,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer3,
    device=device,
    criterion=criterion,
    num_epochs=30,
    patience=10,
    dynamic_sigma_reduction=False,
    resolution_increasing=False,
)

## Models Evaluation
The primary evaluation metric is **pixel localization error**:

- Euclidean distance (in pixels) between:
  - Ground truth heatmap argmax
  - Predicted heatmap argmax

Reported metrics include:
- Per-level mean pixel error
- Overall mean pixel error
- Overall 90th percentile (p90) pixel error

“Among the evaluated architectures, Res-UNet achieves the best trade-off between accuracy and robustness, reducing both mean and extreme pixel localization errors across all vertebral levels.”

In [ ]:
# Validation
table_val, rows_val = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=val_loader,
    device=device,
    use_amp=True,
)

In [ ]:
# Test
table_test, rows_test = compare_models_pixel_error_heatmap(
    models=[model1, model2, model3],
    model_names=["UNet", "Res-UNet", "Attention-Res-UNet"],
    loader=test_loader,
    device=device,
    use_amp=True,
)

### Best models performance visualization

In [ ]:
mean_t = torch.tensor(mean, dtype=torch.float32)
std_t  = torch.tensor(std,  dtype=torch.float32)

for idx in range(15):
    print(f"\nVisualizing val sample {idx}/{len(val_ds)-1}")

    visualize_sample(
        model=model2,
        dataset= test_ds,
        idx=idx,
        device=device,
        mean=mean_t,
        std=std_t,
    )